In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
import shutil
import os

zip_path = '/content/drive-download-20260612T140644Z-3-001.zip'
extract_path = '/content/extracted_datasets'
drive_dest = '/content/drive/MyDrive/diss-datasets'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

os.makedirs(drive_dest, exist_ok=True)
for root, dirs, files in os.walk(extract_path):
    for f in files:
        shutil.copy(os.path.join(root, f), drive_dest)

In [ ]:
import os
for f in os.listdir(drive_dest):
    print(f)

In [ ]:
!pip install fABBA transformers torch scikit-learn --quiet

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from fABBA import fABBA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

embed_model_name = 'gpt2'
tokenizer = AutoTokenizer.from_pretrained(embed_model_name)
tokenizer.pad_token = tokenizer.eos_token
embed_model = AutoModel.from_pretrained(embed_model_name).to(device)
embed_model.eval()

def channel_to_symbolic(channel_series, tol=0.1, alpha=0.1):
    fabba = fABBA(tol=tol, alpha=alpha, sorting='2-norm', scl=1, verbose=0)
    try:
        return fabba.fit_transform(list(channel_series))
    except Exception:
        return 'A'

def sample_to_text(sample, tol=0.1, alpha=0.1):
    n_channels = sample.shape[0]
    parts = [f"channel{c}: {channel_to_symbolic(sample[c], tol, alpha)}" for c in range(n_channels)]
    return ' '.join(parts)

@torch.no_grad()
def embed_text(text):
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512, padding=True).to(device)
    outputs = embed_model(**inputs)
    mask = inputs['attention_mask'].unsqueeze(-1)
    pooled = (outputs.last_hidden_state * mask).sum(1) / mask.sum(1)
    return pooled.squeeze(0).cpu().numpy()

def extract_embeddings(X, tol=0.1, alpha=0.1):
    return np.array([embed_text(sample_to_text(X[i], tol, alpha)) for i in range(X.shape[0])])

def run_linear_probe(X_train, y_train, X_test, y_test, tol=0.1, alpha=0.1):
    train_emb = extract_embeddings(X_train, tol, alpha)
    test_emb = extract_embeddings(X_test, tol, alpha)
    scaler = StandardScaler()
    train_emb_scaled = scaler.fit_transform(train_emb)
    test_emb_scaled = scaler.transform(test_emb)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(train_emb_scaled, y_train)
    preds = clf.predict(test_emb_scaled)
    acc = (preds == y_test).mean()
    return acc, train_emb, test_emb

In [ ]:
dataset_name = 'CMJ'
data = np.load(os.path.join(drive_dest, f'{dataset_name}.npy'), allow_pickle=True).item()
acc, train_emb, test_emb = run_linear_probe(data['train']['X'], data['train']['y'], data['test']['X'], data['test']['y'])
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_train.npy'), train_emb)
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_test.npy'), test_emb)
with open(os.path.join(drive_dest, f'{dataset_name}_text_results.txt'), 'w') as f:
    f.write(f'accuracy: {acc}')
print(dataset_name, 'accuracy:', acc)

In [ ]:
dataset_name = 'MP8'
data = np.load(os.path.join(drive_dest, f'{dataset_name}.npy'), allow_pickle=True).item()
acc, train_emb, test_emb = run_linear_probe(data['train']['X'], data['train']['y'], data['test']['X'], data['test']['y'])
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_train.npy'), train_emb)
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_test.npy'), test_emb)
with open(os.path.join(drive_dest, f'{dataset_name}_text_results.txt'), 'w') as f:
    f.write(f'accuracy: {acc}')
print(dataset_name, 'accuracy:', acc)

In [ ]:
dataset_name = 'MP50'
data = np.load(os.path.join(drive_dest, f'{dataset_name}.npy'), allow_pickle=True).item()
acc, train_emb, test_emb = run_linear_probe(data['train']['X'], data['train']['y'], data['test']['X'], data['test']['y'])
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_train.npy'), train_emb)
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_test.npy'), test_emb)
with open(os.path.join(drive_dest, f'{dataset_name}_text_results.txt'), 'w') as f:
    f.write(f'accuracy: {acc}')
print(dataset_name, 'accuracy:', acc)

In [ ]:
dataset_name = 'SYNTH'
data = np.load(os.path.join(drive_dest, f'{dataset_name}.npy'), allow_pickle=True).item()
acc, train_emb, test_emb = run_linear_probe(data['train']['X'], data['train']['y'], data['test']['X'], data['test']['y'])
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_train.npy'), train_emb)
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_test.npy'), test_emb)
with open(os.path.join(drive_dest, f'{dataset_name}_text_results.txt'), 'w') as f:
    f.write(f'accuracy: {acc}')
print(dataset_name, 'accuracy:', acc)

In [ ]:
dataset_name = 'Rowing'
data = np.load(os.path.join(drive_dest, f'{dataset_name}.npy'), allow_pickle=True).item()
acc, train_emb, test_emb = run_linear_probe(data['train']['X'], data['train']['y'], data['test']['X'], data['test']['y'])
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_train.npy'), train_emb)
np.save(os.path.join(drive_dest, f'{dataset_name}_text_embeddings_test.npy'), test_emb)
with open(os.path.join(drive_dest, f'{dataset_name}_text_results.txt'), 'w') as f:
    f.write(f'accuracy: {acc}')
print(dataset_name, 'accuracy:', acc)